# Validação do Dataset - *Melipona quadrifasciata*
## Treinamento YOLO11s + Avaliação no Conjunto de Teste

---
**Objetivo:** treinar o detector YOLO11s sobre o *dataset* anotado de *M. quadrifasciata* e avaliá-lo no conjunto de teste reservado, gerando as métricas e figuras para a Seção 4.4 da dissertação.

**Características:**
- Variante única: **YOLO11s** (300 épocas)
- Retomada automática de checkpoint (Google Drive)
- Avaliação final no *split* de **teste** (não visto no treino)
- Geração automática de: `results.png`, `confusion_matrix.png`, curva PR e `results.csv`
- Exportação consolidada das métricas em `metricas_teste.json`

> Pipeline limpo e rastreável, alinhado ao princípio de reprodutibilidade da dissertação.

## CÉLULA 1 - Setup (GPU + Ultralytics)

In [ ]:
print("Verificando GPU e instalando dependências...\n")
!nvidia-smi

!pip install ultralytics -q

from ultralytics import YOLO
import os, glob, json, shutil
from pathlib import Path

import ultralytics
print(f"\nUltralytics {ultralytics.__version__}")
print(f" GPU disponível: {'Sim' if os.path.exists('/proc/driver/nvidia/version') else 'Não'}")

## CÉLULA 2 - Montar Google Drive

In [ ]:
from google.colab import drive, files

print("Montando Google Drive...")
drive.mount('/content/drive')

DRIVE_PATH = os.environ.get('MQ_DRIVE_PATH', '/content/drive/MyDrive/mandacaia_yolo_training')
os.makedirs(f'{DRIVE_PATH}/datasets', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/models', exist_ok=True)

RUN_NAME = 'yolo11s_300epochs'
MODEL_DIR = f'{DRIVE_PATH}/models/{RUN_NAME}'

print(f"Drive montado. Pasta de trabalho: {DRIVE_PATH}")

# Detectar checkpoint para retomada
last_ckpt = f'{MODEL_DIR}/weights/last.pt'
if os.path.exists(last_ckpt):
    print(f"\n Checkpoint encontrado - o treino será RETOMADO:\n   {last_ckpt}")
else:
    print("\nNenhum checkpoint. O treino iniciará do zero.")

## CÉLULA 3 - Carregar o Dataset
Usa o *backup* existente no Drive, ou faz *upload* de um `.zip` na primeira execução.

> O `.zip` deve conter a estrutura YOLO: `train/`, `valid/`, `test/` (cada um com `images/` e `labels/`) e o `data.yaml`.

In [ ]:
BACKUP_DIR = f'{DRIVE_PATH}/datasets/dataset_backup'

if os.path.exists(f'{BACKUP_DIR}/data.yaml'):
    print("Dataset encontrado no Drive. Copiando para /content...")
    shutil.copytree(BACKUP_DIR, 'dataset', dirs_exist_ok=True)
else:
    print("Nenhum dataset no Drive. Faça upload do ZIP...")
    uploaded = files.upload()
    zip_filename = list(uploaded.keys())[0]
    print(f"Recebido: {zip_filename}. Descompactando...")
    !unzip -q "{zip_filename}" -d dataset/
    # Backup no Drive
    src_yaml = glob.glob('dataset/**/data.yaml', recursive=True)[0]
    src_root = os.path.dirname(src_yaml)
    shutil.copytree(src_root, BACKUP_DIR, dirs_exist_ok=True)
    print("Backup salvo no Drive.")

# Localizar data.yaml
yaml_files = glob.glob('dataset/**/data.yaml', recursive=True)
assert yaml_files, "data.yaml não encontrado! Verifique a estrutura do ZIP."
DATA_YAML = yaml_files[0]
DATASET_DIR = os.path.dirname(DATA_YAML)

print(f"\ndata.yaml: {DATA_YAML}\n")
print("Conteúdo:")
!cat "{DATA_YAML}"

# Contagem por split
for split in ['train', 'valid', 'test']:
    n = len(glob.glob(f'{DATASET_DIR}/{split}/images/*.*'))
    print(f"   {split:<6}: {n} imagens")

## CÉLULA 4 - Treinar YOLO11s (300 épocas)
Detecta automaticamente o checkpoint e retoma de onde parou.

> Hiperparâmetros idênticos aos da configuração original (batch=12, imgsz=640, augmentações para abelhas).

In [ ]:
print("="*70)
print("TREINAMENTO - YOLO11s (300 épocas)")
print("="*70)

if os.path.exists(last_ckpt):
    print(f"Retomando de: {last_ckpt}")
    model = YOLO(last_ckpt)
    resume = True
else:
    print("Iniciando do zero com pesos pré-treinados COCO (yolo11s.pt)")
    model = YOLO('yolo11s.pt')
    resume = False

results = model.train(
    data=DATA_YAML,
    epochs=300,
    imgsz=640,
    batch=12,
    patience=50,
    device=0,
    project=f'{DRIVE_PATH}/models',
    name=RUN_NAME,
    exist_ok=True,
    resume=resume,
    save=True,
    save_period=15,
    # Augmentações
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15, translate=0.1, scale=0.5,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.0,
    # Otimização
    optimizer='auto',
    lr0=0.01, lrf=0.005, cos_lr=True,
    warmup_epochs=5, warmup_momentum=0.8, weight_decay=0.0005,
    plots=True, verbose=True,
)

print("\nTreino concluído!")
print(f"Resultados: {MODEL_DIR}/")

## CÉLULA 5 - Avaliação no Conjunto de TESTE
Esta é a métrica oficial da dissertação: o conjunto de teste foi separado por colônia e por similaridade semântica (CLIP), e **não foi visto durante o treino**.

In [ ]:
print("="*70)
print("AVALIAÇÃO NO CONJUNTO DE TESTE - YOLO11s")
print("="*70)

best = YOLO(f'{MODEL_DIR}/weights/best.pt')

# Avaliar no split de teste
metrics = best.val(
    data=DATA_YAML,
    split='test',
    project=f'{DRIVE_PATH}/models',
    name=f'{RUN_NAME}_TEST',
    exist_ok=True,
    plots=True,   # gera matriz de confusão e curva PR no teste
)

# Métricas globais
p   = float(metrics.box.mp)
r   = float(metrics.box.mr)
m50 = float(metrics.box.map50)
m95 = float(metrics.box.map)
f1  = 2*p*r/(p+r) if (p+r) > 0 else 0.0

print(f"\n{'='*70}")
print("MÉTRICAS GLOBAIS NO TESTE (YOLO11s, 300 épocas)")
print(f"{'='*70}")
print(f"  mAP@50:      {m50:.4f}  ({m50*100:.2f}%)")
print(f"  mAP@50-95:   {m95:.4f}  ({m95*100:.2f}%)")
print(f"  Precision:   {p:.4f}  ({p*100:.2f}%)")
print(f"  Recall:      {r:.4f}  ({r*100:.2f}%)")
print(f"  F1-Score:    {f1:.4f}")
print(f"{'='*70}")

# Métricas por classe
print("\nMÉTRICAS POR CLASSE:")
names = best.names  # {0: 'mandacaia', 1: 'entrada'} ou conforme data.yaml
per_class = {}
for i, c in enumerate(metrics.box.ap_class_index):
    cls_name = names[int(c)]
    cls_p, cls_r, cls_ap50, cls_ap = metrics.box.class_result(i)
    per_class[cls_name] = {
        'precision': round(float(cls_p), 4),
        'recall':    round(float(cls_r), 4),
        'map50':     round(float(cls_ap50), 4),
        'map50_95':  round(float(cls_ap), 4),
    }
    print(f"  {cls_name:<12} P={cls_p:.4f}  R={cls_r:.4f}  mAP50={cls_ap50:.4f}  mAP50-95={cls_ap:.4f}")

## CÉLULA 6 - Consolidar e Exportar Métricas
Salva tudo em `metricas_teste.json` no Drive. **É este arquivo que você me envia.**

In [ ]:
import datetime

# Velocidade de inferência (ms/imagem) reportada pelo validador
speed = getattr(metrics, 'speed', {})

export = {
    'modelo': 'YOLO11s',
    'epocas': 300,
    'imgsz': 640,
    'batch': 12,
    'data_avaliacao': datetime.datetime.now().isoformat(),
    'ultralytics_version': ultralytics.__version__,
    'split': 'test',
    'metricas_globais': {
        'precision': round(p, 4),
        'recall':    round(r, 4),
        'map50':     round(m50, 4),
        'map50_95':  round(m95, 4),
        'f1':        round(f1, 4),
    },
    'metricas_por_classe': per_class,
    'velocidade_ms': {k: round(float(v), 2) for k, v in speed.items()},
}

out_json = f'{MODEL_DIR}/metricas_teste.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(export, f, indent=2, ensure_ascii=False)

print("Métricas exportadas:")
print(json.dumps(export, indent=2, ensure_ascii=False))
print(f"\nSalvo em: {out_json}")

# Baixar direto para a máquina local
try:
    files.download(out_json)
except Exception as e:
    print(f"(Baixe manualmente do Drive se o download automático falhar: {e})")

## CÉLULA 7 - Localizar as Figuras Geradas
Lista os caminhos das figuras de treino (validação) e de teste para você baixar e me enviar.

In [ ]:
from IPython.display import Image, display

test_dir = f'{DRIVE_PATH}/models/{RUN_NAME}_TEST'

print("FIGURAS DO TREINO/VALIDAÇÃO (pasta do modelo):")
for fig in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
            'PR_curve.png', 'F1_curve.png']:
    path = f'{MODEL_DIR}/{fig}'
    print(f"   {'' if os.path.exists(path) else ''} {path}")

print("\nFIGURAS DA AVALIAÇÃO NO TESTE:")
for fig in ['confusion_matrix.png', 'confusion_matrix_normalized.png',
            'PR_curve.png', 'F1_curve.png']:
    path = f'{test_dir}/{fig}'
    print(f"   {'' if os.path.exists(path) else ''} {path}")

# Exibir as principais inline
print("\n--- Curva de treino (results.png) ---")
if os.path.exists(f'{MODEL_DIR}/results.png'):
    display(Image(filename=f'{MODEL_DIR}/results.png'))

print("\n--- Matriz de confusão no TESTE ---")
if os.path.exists(f'{test_dir}/confusion_matrix.png'):
    display(Image(filename=f'{test_dir}/confusion_matrix.png'))

print("\n--- Curva Precisão-Revocação no TESTE ---")
if os.path.exists(f'{test_dir}/PR_curve.png'):
    display(Image(filename=f'{test_dir}/PR_curve.png'))

print("\nBaixe as figuras marcadas com e envie junto com metricas_teste.json")